## Object Detection Project: Fine-Tune YOLO on a Custom Dataset

In the object detection notebook, you saw YOLO fine-tuned on African Wildlife and NEU Steel Defects. Same pretrained model, same workflow, completely different domains. Now it's your turn to apply this to a new dataset.

The goal: download a dataset, fine-tune YOLO, evaluate your model, and understand what it gets right and where it struggles. Detection is one of the most commercially deployed ML tasks - the workflow you'll practice here is the same one used in production systems.

In [ ]:
# Setup - run this first
try:
    import google.colab
    IN_COLAB = True
    !pip install -q ultralytics
except ImportError:
    IN_COLAB = False

from ultralytics import YOLO
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

## Pick a Dataset

All datasets below are built into Ultralytics - they auto-download when you reference their YAML file. No signup, no API keys.

### Construction PPE (recommended)

**What it is:** 1,416 images of construction workers. The model detects whether workers are wearing required safety equipment.

**Classes (11):** Helmet, Gloves, Vest, Boots, Goggles, Person, no_helmet, no_goggle, no_gloves, no_boots, none

**Loading:** `data='construction-ppe.yaml'`

| Pros | Cons |
|------|------|
| Real industry use case - workplace safety monitoring | 11 classes is more than the 4 you practiced with |
| Detects both presence AND absence of PPE (interesting twist) | Some classes defined by absence are harder |
| Same workflow as African Wildlife - just swap the yaml | 178 MB download |
| No signup needed | |

### Brain Tumor Detection

**What it is:** 1,116 brain MRI images. The model localizes tumors.

**Classes (2):** Negative, Positive

**Loading:** `data='brain-tumor.yaml'`

| Pros | Cons |
|------|------|
| Medical imaging - compelling and impactful domain | Only 2 classes - simpler task |
| Tiny download (4 MB) - very fast iteration | Less variety in detection challenges |
| Shows YOLO beyond natural photos | MRI domain is very different from COCO pretraining |

### African Wildlife

**What it is:** 1,052 images of African animals. Same dataset from the lesson.

**Classes (4):** Buffalo, Elephant, Rhino, Zebra

**Loading:** `data='african-wildlife.yaml'`

| Pros | Cons |
|------|------|
| Familiar from class - you know what to expect | You've already seen the results |
| 4 classes is manageable | Less learning from repeating the same dataset |
| Good if you want to focus on experimentation over data setup | |

## Recommendations

### Top pick: Construction PPE

This is one of the most common real-world YOLO deployments. Companies like Intenseye and Voxel AI build products around exactly this. 11 classes is a step up from the 4-class wildlife example, and the "has PPE" vs "missing PPE" distinction (helmet vs no_helmet) adds an interesting challenge - detecting the absence of something is harder than detecting its presence.

### If you want something different: Brain Tumor

Medical imaging is a completely different visual domain. Only 2 classes makes the detection simpler, but the domain shift from COCO (everyday objects) to MRI scans is a great test of transfer learning limits.

## Project Tasks

### Phase 1: Explore the Data

*Goal: Understand what you're detecting before training.*

**Task 1.1:** Download your dataset by running a quick validation pass:
```python
model = YOLO('yolo11n.pt')
model.val(data='your-dataset.yaml', verbose=False, plots=False)
```

**Task 1.2:** Find and print the `dataset.yaml` file. What are the class names? How many training/validation images?

**Task 1.3:** Open a few annotation `.txt` files. Pick an image with multiple annotations and draw the ground-truth bounding boxes on it (use `matplotlib.patches.Rectangle` - you saw this in the lesson notebook).

**Task 1.4:** Count how many annotations exist per class. Is the dataset balanced?

### Phase 2: Pretrained Baseline

*Goal: See what COCO pretraining gives you before any fine-tuning.*

**Task 2.1:** Run the pretrained YOLO model (no fine-tuning) on 5-10 of your dataset's images. What does it detect? What does it miss? Some COCO classes might overlap with your dataset's classes.

**Task 2.2:** Try different confidence thresholds (0.1, 0.25, 0.5) on the same image. How does the output change?

### Phase 3: Fine-Tune

*Goal: Train YOLO on your specific domain.*

**Task 3.1:** Fine-tune YOLO11-nano:
```python
model = YOLO('yolo11n.pt')
results = model.train(data='your-dataset.yaml', epochs=15, imgsz=640, batch=16)
```

**Task 3.2:** Monitor the training output. Does the loss decrease steadily? Any signs of overfitting?

### Phase 4: Evaluate

*Goal: Measure how well your fine-tuned model performs.*

**Task 4.1:** Run validation and print the metrics:
```python
metrics = model.val(data='your-dataset.yaml')
print(f'mAP@50: {metrics.box.map50:.3f}')
print(f'mAP@50-95: {metrics.box.map:.3f}')
print(f'Precision: {metrics.box.mp:.3f}')
print(f'Recall: {metrics.box.mr:.3f}')
```

**Task 4.2:** What do these metrics mean? mAP@50 measures detection quality at IoU threshold 0.5. mAP@50-95 averages across thresholds 0.5 to 0.95 (much stricter). Which is more relevant for your use case?

### Phase 5: Inspect Predictions

*Goal: Understand what the model gets right and where it struggles.*

**Task 5.1:** Run the fine-tuned model on all validation images. Show 4-6 images where the model is most confident (high average confidence, multiple detections).

**Task 5.2:** Show 4-6 images where the model struggles (low confidence, missed detections, or no detections at all).

**Task 5.3:** Compare the pretrained COCO model vs your fine-tuned model on the same images (side by side). What changed?

**Task 5.4:** List the failure modes you observe. Are there patterns? (Small objects? Occlusion? Unusual angles? Specific classes?)

### Phase 6: Experiment (Optional)

*Goal: Try to improve your results.*

**Task 6.1:** Train for more epochs (25-30). Does accuracy keep improving or does it plateau?

**Task 6.2:** Try a larger model: `YOLO('yolo11s.pt')` instead of nano. How does it compare in accuracy and speed?

**Task 6.3:** Try a different image size: `imgsz=320` (faster) vs `imgsz=640` (default) vs `imgsz=1024` (if your GPU can handle it). How does this affect detection of small objects?

### Phase 7: Reflection

**Task 7.1:** Build a summary of your results:

| | mAP@50 | mAP@50-95 | Precision | Recall |
| --- | --- | --- | --- | --- |
| COCO pretrained (no fine-tuning) | | | | |
| Fine-tuned (your best) | | | | |

**Task 7.2:** If you were deploying this model in production (e.g. a camera on a factory floor or a construction site), what would concern you? Think about: failure modes, confidence thresholds, edge cases, false positives vs false negatives.

**Task 7.3:** How is this workflow similar to the classification fine-tuning from the other homework? How is it different?

<div style="text-align: center; color: #888; font-size: 0.85em; margin-top: 40px; padding-top: 10px; border-top: 1px solid #ddd;">
&copy; 2025 Utvecklarakademin UA Aktiebolag. All rights reserved.<br>
This material is proprietary and may not be reproduced, distributed, or shared without written permission.
</div>